<a href="https://colab.research.google.com/github/angelsnelson/angelsnelson/blob/main/AI_P1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PRACTICUM 1

Import Libraries

In [1]:
import kagglehub
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn import svm

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

Download the Dataset

In [2]:
path = kagglehub.dataset_download(
    "uciml/sms-spam-collection-dataset"
)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Path to dataset files: /kaggle/input/sms-spam-collection-dataset


Check the Dataset Files

In [3]:
print(os.listdir(path))

['spam.csv']


Load the Dataset

In [4]:
file_path = os.path.join(path, "spam.csv")

df = pd.read_csv(
    file_path,
    encoding="latin-1"
)

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


Keep Only the Required Columns

In [5]:
df = df[["v1", "v2"]]

df.columns = ["label", "message"]

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Check Dataset Information

In [6]:
print("Dataset Shape:", df.shape)

print("\nLabel Count:")
print(df["label"].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (5572, 2)

Label Count:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing Values:
label      0
message    0
dtype: int64


Convert Labels into Numbers

In [7]:
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


Define Input and Target

In [8]:
X = df["message"]
y = df["label"]

print("Input Data:")
print(X.head())

print("\nTarget:")
print(y.head())

Input Data:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: object

Target:
0    0
1    0
2    1
3    0
4    0
Name: label, dtype: int64


Split the Dataset

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Total Data:", len(X))
print("Training Data:", len(X_train))
print("Testing Data:", len(X_test))

Total Data: 5572
Training Data: 4457
Testing Data: 1115


View Training Data

In [10]:
print("Training Messages:")
print(X_train.head())

print("\nTraining Labels:")
print(y_train.head())

Training Messages:
184                            Going on nothing great.bye
2171                        I wont. So wat's wit the guys
5422              Ok k..sry i knw 2 siva..tats y i askd..
4113    Where are you ? What do you do ? How can you s...
4588         Have you not finished work yet or something?
Name: message, dtype: object

Training Labels:
184     0
2171    0
5422    0
4113    0
4588    0
Name: label, dtype: int64


TF-IDF Vectorization

In [11]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000
)

X_train_vector = vectorizer.fit_transform(X_train)

X_test_vector = vectorizer.transform(X_test)

Check Vectorized Data

In [12]:
print("Training Vector Shape:")
print(X_train_vector.shape)

print("\nTesting Vector Shape:")
print(X_test_vector.shape)

print("\nOriginal Message:")
print(X_train.iloc[0])

print("\nVectorized Message:")
print(X_train_vector[0])

Training Vector Shape:
(4457, 5000)

Testing Vector Shape:
(1115, 5000)

Original Message:
Going on nothing great.bye

Vectorized Message:
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (1, 5000)>
  Coords	Values
  (0, 1885)	0.442087724329911
  (0, 1946)	0.4873194928223273
  (0, 693)	0.7530459188603245


Create the SVM Model

In [13]:
model = svm.SVC(
    kernel="linear"
)

model

SVC(kernel='linear')

Train the SVM Model

In [14]:
model.fit(
    X_train_vector,
    y_train
)

print("Model training completed.")

Model training completed.


Predict the Test Data

In [15]:
y_pred = model.predict(
    X_test_vector
)

print(y_pred[:20])

[0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


Compare Actual and Predicted Values

In [16]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results.head(20)

,Actual,Predicted
0,0,0
1,0,0
2,0,0
3,1,1
4,0,0
5,0,0
6,0,0
7,0,0
8,0,0
9,0,0


Calculate Accuracy

In [17]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 0.9847533632286996
Accuracy: 98.48 %


Classification Report

In [18]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Ham", "Spam"]
    )
)

              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       966
        Spam       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.98      0.98      0.98      1115



Confusion Matrix

In [19]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[965   1]
 [ 16 133]]


Testing with Own SMS

In [20]:
new_message = [
    "Congratulations! You have won a free prize. Call now!"
]

new_vector = vectorizer.transform(
    new_message
)

prediction = model.predict(
    new_vector
)

print(prediction)

[1]


Display Spam or Ham

In [21]:
if prediction[0] == 1:
    print("SPAM MESSAGE")
else:
    print("HAM MESSAGE")

SPAM MESSAGE


Test Another Normal Message

In [22]:
new_message = [
    "Hi, are you coming to college today?"
]

new_vector = vectorizer.transform(
    new_message
)

prediction = model.predict(
    new_vector
)

if prediction[0] == 1:
    print("SPAM")
else:
    print("HAM")

HAM
